# Tokenix — does a tokenizer-graph penalty help training?

This notebook trains a ~51M-parameter GPT (GPT-2 tokenizer, tied embeddings) on FineWeb-Edu.
The runs differ only in a penalty on the token embedding `E`:

| condition | loss |
|---|---|
| `baseline` | cross-entropy |
| `prefix` | + λ · Dirichlet energy of `E` on the **prefix graph** (trie parent + ` t`↔`t`) |
| `prefix:shuffled` | same penalty, graph vertices randomly relabelled (**the control**) |
| `contain` / `contain:shuffled` | same with the substring-poset graph |

The claim holds only if **`prefix` beats both `baseline` and `prefix:shuffled`** across seeds.

Everything (data, checkpoints, logs) lives on Google Drive, so a disconnected session resumes where it stopped: re-run all cells.

**Runtime → Change runtime type → A100 GPU.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/tokenix'   # data + runs are kept here
BRANCH = 'main'                           # or a feature branch
import os
os.makedirs(WORK, exist_ok=True)

In [ ]:
%cd /content
!rm -rf Tokenix && git clone -q --branch $BRANCH https://github.com/TheFausap/Tokenix
%cd /content/Tokenix
!pip -q install -e . datasets tokenizers matplotlib
!git log --oneline -1

## 1. Data (once, ~20–30 min; cached on Drive)

500M training tokens + 5M validation tokens of `HuggingFaceFW/fineweb-edu` (`sample-10BT`), GPT-2 tokenizer, ~1 GB on Drive.
Then copied to the local disk, because random reads from Drive are slow.

In [ ]:
DRIVE_DATA = f'{WORK}/data/fineweb_gpt2'
if not os.path.exists(f'{DRIVE_DATA}/meta.json'):
    !python experiments/prepare_data.py --out {DRIVE_DATA} --train_tokens 500000000 --val_tokens 5000000
DATA = '/content/data'
!mkdir -p {DATA} && rsync -a --info=progress2 {DRIVE_DATA}/ {DATA}/
!ls -la {DATA}

## 2. Throughput check (~3 min)

Checks memory and speed before committing hours. Expect roughly 250–400k tokens/s on an A100 with `torch.compile`.
If you hit out-of-memory, halve `--batch` and double `--accum` (same tokens per step).

In [ ]:
!python experiments/train_gpt.py --data {DATA} --out /content/smoke --condition prefix --tokens 3e7 \
    --eval_every 50 --eval_iters 5 --name smoke 2>&1 | grep -v Warning | tail -5

## 3. Pick λ (optional, ~40 min)

Short runs (100M tokens) of `prefix` and its shuffled control at several λ. Choose the largest λ at which
the real graph is not worse than baseline; the main grid uses it for **both** the real and the shuffled graph.

In [ ]:
PILOT = f'{WORK}/pilot'
for cond in ['baseline'] + [f'prefix{m}' for m in ['', ':shuffled']]:
    lams = [0.0] if cond == 'baseline' else [0.03, 0.1, 0.3]
    for lam in lams:
        !python experiments/train_gpt.py --data {DATA} --out {PILOT} --condition {cond} --lam {lam} --tokens 1e8 2>&1 | grep -v Warning | tail -1
!python experiments/summarize_runs.py {PILOT}

## 4. Main grid (~30–35 min per run → ~5–6 h for 10 runs)

Finished runs are skipped and interrupted ones resume from their last checkpoint, so after a disconnect
just re-run the cells above (skipping 2–3) and this one.

In [ ]:
LAM = 0.1          # set from step 3
SEEDS = [0, 1]
CONDITIONS = ['baseline', 'prefix', 'prefix:shuffled', 'contain', 'contain:shuffled']
RUNS = f'{WORK}/runs'

for seed in SEEDS:
    for cond in CONDITIONS:
        name = cond.replace(':', '-') + ('' if cond == 'baseline' else f'_lam{LAM:g}') + f'_s{seed}'
        if os.path.exists(f'{RUNS}/{name}/DONE'):
            print('done   ', name); continue
        print('running', name)
        !python experiments/train_gpt.py --data {DATA} --out {RUNS} --condition {cond} --lam {LAM} --seed {seed} 2>&1 | grep -v Warning | tail -2

## 5. Results

In [ ]:
!python experiments/summarize_runs.py {RUNS} --plot {RUNS}/curves.png
from IPython.display import Image
Image(f'{RUNS}/curves.png')

**Reading the table**

* `Δ vs base` — final validation loss minus baseline (negative = better). Compare with the ± across seeds.
* `tokens→base` — fraction of the baseline's token budget a run needed to reach the baseline's final loss (< 1 = faster).
* `Dir(prefix)`, `Dir(contain)` — the embedding's Dirichlet ratio on each graph (1 = no structure).
  Pretrained GPT-2 / Pythia / Llama sit around 0.78 on the substring graph without any penalty.

The logs (`runs/*/log.jsonl`) are small; share `runs/` minus the `ckpt.pt` files to have them analysed.